# Fase 5.1 — FastAPI

Esta versión productiviza los motores finales del TFM:

- **Predicción:** XGBoost + variables geoespaciales + SHAP.
- **Consultor documental:** retrieval v2.2 + prompt v3 + `mistral:7b`.

Genera `src/rag_utils.py`, `api.py` y `src/api_client.py`.

## 1. Configuración

El RAG reutiliza los artefactos finales de la Fase 4 y no vuelve a procesar
los PDF ni reconstruye ChromaDB.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"

RAG_UTILS_PATH = SRC_DIR / "rag_utils.py"
API_PATH = PROJECT_ROOT / "api.py"
API_CLIENT_PATH = SRC_DIR / "api_client.py"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAG_UTILS_PATH:", RAG_UTILS_PATH)
print("API_PATH:", API_PATH)
print("API_CLIENT_PATH:", API_CLIENT_PATH)

## 2. Generar `src/rag_utils.py`

Esta celda genera la versión base del módulo RAG utilizada durante la
productivización, a partir del retrieval v2.2, el prompt v3, top-k 4
y `mistral:7b`.

La versión operativa final de `src/rag_utils.py` incorpora ajustes posteriores
realizados durante las pruebas de integración.

In [ ]:
RAG_UTILS_CODE = 'from pathlib import Path\nimport re\nimport time\nimport unicodedata\nimport warnings\n\nimport chromadb\nimport numpy as np\nimport pandas as pd\nfrom sentence_transformers import SentenceTransformer\nfrom langchain_core.prompts import ChatPromptTemplate\nfrom langchain_ollama import ChatOllama\n\nwarnings.filterwarnings("ignore")\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\n\nPROCESSED_RAG_DIR = (\n    PROJECT_ROOT / "data" / "processed" / "rag_v2"\n)\n\nVECTOR_DB_DIR = (\n    PROJECT_ROOT / "data" / "vectorstore" / "chroma_vut_malaga_v2"\n)\n\nCHUNKS_PATH = (\n    PROCESSED_RAG_DIR / "rag_chunks_v2.parquet"\n)\n\nEMBEDDING_MODEL_NAME = (\n    "paraphrase-multilingual-MiniLM-L12-v2"\n)\n\nCOLLECTION_NAME = (\n    "vut_malaga_normativa_v2"\n)\n\nFINAL_LLM = "mistral:7b"\n\nCANDIDATE_POOL = 15\nFINAL_TOP_K = 4\nMIN_SIMILARITY = 0.28\nNEAR_DUPLICATE_THRESHOLD = 0.93\nMMR_LAMBDA = 0.72\nMAX_PER_DOCUMENT = 2\n\nif not CHUNKS_PATH.exists():\n    raise FileNotFoundError(\n        f"No existe el fichero de chunks: {CHUNKS_PATH}"\n    )\n\nif not VECTOR_DB_DIR.exists():\n    raise FileNotFoundError(\n        f"No existe el vectorstore: {VECTOR_DB_DIR}"\n    )\n\nchunks_df = pd.read_parquet(CHUNKS_PATH)\n\nembedding_model = SentenceTransformer(\n    EMBEDDING_MODEL_NAME\n)\n\nchroma_client = chromadb.PersistentClient(\n    path=str(VECTOR_DB_DIR)\n)\n\ncollection = chroma_client.get_collection(\n    COLLECTION_NAME\n)\n\n\n\nDOCUMENT_ALIASES = {\n    "Decreto 31/2024": [\n        "decreto 31/2024",\n        "decreto 31 2024",\n        "decreto 31",\n    ],\n    "Decreto 28/2016": [\n        "decreto 28/2016",\n        "decreto 28 2016",\n        "decreto 28",\n    ],\n    "Instrucción 1/2024": [\n        "instrucción 1/2024",\n        "instruccion 1/2024",\n        "instrucción 1 2024",\n        "instruccion 1 2024",\n    ],\n    "Resolución sobre Instrucción 1/2024": [\n        "resolución",\n        "resolucion",\n    ],\n    "Informe jurídico sobre Instrucción 1/2024": [\n        "informe jurídico",\n        "informe juridico",\n    ],\n    "Memoria modificación PGOU VUT": [\n        "memoria",\n        "pgou",\n    ],\n    "Resumen Ejecutivo PGOU VUT": [\n        "resumen ejecutivo",\n    ],\n    "Acuerdo Pleno aprobación definitiva": [\n        "acuerdo pleno",\n        "acuerdo del pleno",\n        "aprobación definitiva",\n        "aprobacion definitiva",\n    ],\n    "Informe de Impacto de la Vivienda Turística": [\n        "informe de impacto",\n        "impacto de la vivienda turística",\n        "impacto de la vivienda turistica",\n    ],\n}\n\n\ndef normalize_text(text):\n    text = unicodedata.normalize(\n        "NFKD",\n        str(text),\n    )\n\n    text = "".join(\n        char\n        for char in text\n        if not unicodedata.combining(\n            char\n        )\n    )\n\n    text = text.lower()\n\n    text = re.sub(\n        r"\\s+",\n        " ",\n        text,\n    )\n\n    return text.strip()\n\n\ndef detect_explicit_documents(query):\n    query_normalized = normalize_text(\n        query\n    )\n\n    detected = []\n\n    for title, aliases in (\n        DOCUMENT_ALIASES.items()\n    ):\n        normalized_aliases = [\n            normalize_text(alias)\n            for alias in aliases\n        ]\n\n        if any(\n            alias in query_normalized\n            for alias in normalized_aliases\n        ):\n            detected.append(title)\n\n    return detected\n\n\ndef cosine_similarity_matrix(matrix):\n    matrix = np.asarray(\n        matrix,\n        dtype=np.float32,\n    )\n\n    norms = np.linalg.norm(\n        matrix,\n        axis=1,\n        keepdims=True,\n    )\n\n    norms = np.clip(\n        norms,\n        1e-12,\n        None,\n    )\n\n    normalized = matrix / norms\n\n    return normalized @ normalized.T\n\n\nCHANGE_TERMS = [\n    "modifica",\n    "modificación",\n    "modificacion",\n    "queda modificado",\n    "queda modificada",\n    "se incorpora",\n    "se incorporan",\n    "se añade",\n    "se anade",\n    "nuevos requisitos",\n    "nuevo requisito",\n    "se sustituye",\n    "se refuerza",\n]\n\nRESIDENTIAL_IMPACT_TERMS = [\n    "impacto",\n    "vivienda residencial",\n    "acceso a la vivienda",\n    "alquiler",\n    "precios de alquiler",\n    "precio de la vivienda",\n    "hogares",\n    "población",\n    "poblacion",\n    "viviendas principales",\n    "viviendas vacías",\n    "viviendas vacias",\n    "presión turística residencial",\n    "presion turistica residencial",\n    "tensionando",\n    "pierden",\n    "desplazamiento",\n]\n\n\ndef is_change_query(query):\n    query_norm = normalize_text(query)\n\n    change_query_terms = [\n        "que cambios",\n        "que cambia",\n        "que modifica",\n        "que modificaciones",\n        "que introduce",\n        "que incorpora",\n    ]\n\n    return any(\n        term in query_norm\n        for term in change_query_terms\n    )\n\n\ndef is_residential_impact_query(query):\n    query_norm = normalize_text(query)\n\n    return (\n        (\n            "impacto" in query_norm\n            or "efecto" in query_norm\n            or "consecuencia" in query_norm\n        )\n        and (\n            "vivienda" in query_norm\n            or "residencial" in query_norm\n            or "alquiler" in query_norm\n        )\n    )\n\n\ndef lexical_change_bonus(text):\n    text_norm = normalize_text(text)\n\n    hits = sum(\n        1\n        for term in CHANGE_TERMS\n        if normalize_text(term) in text_norm\n    )\n\n    return min(\n        0.12,\n        0.03 * hits,\n    )\n\n\ndef lexical_residential_bonus(text):\n    text_norm = normalize_text(text)\n\n    hits = sum(\n        1\n        for term in RESIDENTIAL_IMPACT_TERMS\n        if normalize_text(term) in text_norm\n    )\n\n    # El bonus es deliberadamente moderado:\n    # ayuda a distinguir impacto residencial de regulación,\n    # sin sustituir la similitud semántica.\n    return min(\n        0.18,\n        0.025 * hits,\n    )\n\n\ndef expand_query_for_retrieval(query):\n    """\n    Expansión muy controlada solo para intenciones donde la consulta\n    original es demasiado general para recuperar evidencia concreta.\n    """\n    if is_residential_impact_query(query):\n        return (\n            f"{query} "\n            "impacto sobre vivienda residencial acceso a la vivienda "\n            "precios de alquiler hogares población viviendas principales "\n            "viviendas vacías presión turística residencial"\n        )\n\n    return query\n\n\ndef retrieve_candidates(\n    query,\n    candidate_pool=CANDIDATE_POOL,\n    document_bonus=0.05,\n):\n    retrieval_query = expand_query_for_retrieval(\n        query\n    )\n\n    # En preguntas de impacto residencial ampliamos el pool\n    # porque los mejores fragmentos pueden no estar entre los\n    # primeros resultados de una consulta muy genérica.\n    if is_residential_impact_query(query):\n        candidate_pool = max(\n            candidate_pool,\n            30,\n        )\n\n    query_embedding = embedding_model.encode(\n        [retrieval_query],\n        normalize_embeddings=True,\n    )\n\n    n_candidates = min(\n        candidate_pool,\n        collection.count(),\n    )\n\n    results = collection.query(\n        query_embeddings=query_embedding.tolist(),\n        n_results=n_candidates,\n        include=[\n            "documents",\n            "metadatas",\n            "distances",\n            "embeddings",\n        ],\n    )\n\n    explicit_documents = detect_explicit_documents(\n        query\n    )\n\n    rows = []\n\n    for i in range(\n        len(results["documents"][0])\n    ):\n        metadata = results["metadatas"][0][i]\n        text = results["documents"][0][i]\n\n        distance = float(\n            results["distances"][0][i]\n        )\n\n        semantic_similarity = 1.0 - distance\n\n        explicit_match = (\n            metadata["short_title"]\n            in explicit_documents\n        )\n\n        residential_bonus = (\n            lexical_residential_bonus(text)\n            if is_residential_impact_query(query)\n            else 0.0\n        )\n\n        effective_relevance = (\n            semantic_similarity\n            + (\n                document_bonus\n                if explicit_match\n                else 0.0\n            )\n            + residential_bonus\n        )\n\n        rows.append(\n            {\n                "candidate_rank": i + 1,\n                "documento": metadata[\n                    "short_title"\n                ],\n                "tipo": metadata[\n                    "document_type"\n                ],\n                "pagina": int(\n                    metadata["page"]\n                ),\n                "chunk_index": int(\n                    metadata.get(\n                        "chunk_index",\n                        0,\n                    )\n                ),\n                "texto": text,\n                "distance": distance,\n                "semantic_similarity": (\n                    semantic_similarity\n                ),\n                "lexical_bonus": residential_bonus,\n                "explicit_document_match": (\n                    explicit_match\n                ),\n                "effective_relevance": (\n                    effective_relevance\n                ),\n                "embedding": np.asarray(\n                    results["embeddings"][0][i],\n                    dtype=np.float32,\n                ),\n            }\n        )\n\n    return pd.DataFrame(rows)\n\n\ndef retrieve_explicit_document(\n    query,\n    document_title,\n    n_results=12,\n):\n    """\n    Recuperación directa dentro del documento citado.\n    Para preguntas de cambios/modificaciones añade un bonus léxico.\n    """\n    query_embedding = embedding_model.encode(\n        [query],\n        normalize_embeddings=True,\n    )[0]\n\n    results = collection.query(\n        query_embeddings=[\n            query_embedding.tolist()\n        ],\n        n_results=n_results,\n        where={\n            "short_title": document_title\n        },\n        include=[\n            "documents",\n            "metadatas",\n            "distances",\n            "embeddings",\n        ],\n    )\n\n    rows = []\n    change_query = is_change_query(query)\n\n    for i in range(\n        len(results["documents"][0])\n    ):\n        metadata = results["metadatas"][0][i]\n        text = results["documents"][0][i]\n\n        distance = float(\n            results["distances"][0][i]\n        )\n\n        semantic_similarity = 1.0 - distance\n\n        lexical_bonus = (\n            lexical_change_bonus(text)\n            if change_query\n            else 0.0\n        )\n\n        rows.append(\n            {\n                "candidate_rank": i + 1,\n                "documento": metadata[\n                    "short_title"\n                ],\n                "tipo": metadata[\n                    "document_type"\n                ],\n                "pagina": int(\n                    metadata["page"]\n                ),\n                "chunk_index": int(\n                    metadata.get(\n                        "chunk_index",\n                        0,\n                    )\n                ),\n                "texto": text,\n                "distance": distance,\n                "semantic_similarity": (\n                    semantic_similarity\n                ),\n                "lexical_bonus": lexical_bonus,\n                "explicit_document_match": True,\n                "effective_relevance": (\n                    semantic_similarity\n                    + 0.15\n                    + lexical_bonus\n                ),\n                "embedding": np.asarray(\n                    results["embeddings"][0][i],\n                    dtype=np.float32,\n                ),\n            }\n        )\n\n    return pd.DataFrame(rows)\n\n\ndef get_neighbor_chunk(\n    document_title,\n    page,\n    chunk_index,\n    direction=1,\n):\n    document_chunks = chunks_df.loc[\n        chunks_df["short_title"]\n        == document_title\n    ].copy()\n\n    document_chunks = (\n        document_chunks\n        .sort_values(\n            [\n                "page",\n                "chunk_index",\n            ]\n        )\n        .reset_index(drop=True)\n    )\n\n    current_matches = document_chunks.index[\n        (\n            document_chunks["page"]\n            == page\n        )\n        & (\n            document_chunks["chunk_index"]\n            == chunk_index\n        )\n    ].tolist()\n\n    if not current_matches:\n        return None\n\n    neighbor_position = (\n        current_matches[0]\n        + direction\n    )\n\n    if (\n        neighbor_position < 0\n        or neighbor_position\n        >= len(document_chunks)\n    ):\n        return None\n\n    return document_chunks.iloc[\n        neighbor_position\n    ]\n\n\ndef remove_near_duplicates(\n    candidates,\n    threshold=NEAR_DUPLICATE_THRESHOLD,\n):\n    if candidates.empty:\n        return candidates.copy()\n\n    candidates = (\n        candidates\n        .sort_values(\n            [\n                "effective_relevance",\n                "semantic_similarity",\n            ],\n            ascending=False,\n        )\n        .reset_index(drop=True)\n    )\n\n    kept_rows = []\n    kept_embeddings = []\n\n    for _, row in candidates.iterrows():\n        current_embedding = row[\n            "embedding"\n        ]\n\n        if not kept_embeddings:\n            kept_rows.append(\n                row.to_dict()\n            )\n            kept_embeddings.append(\n                current_embedding\n            )\n            continue\n\n        similarities = [\n            float(\n                np.dot(\n                    current_embedding,\n                    previous_embedding,\n                )\n            )\n            for previous_embedding\n            in kept_embeddings\n        ]\n\n        if max(similarities) < threshold:\n            kept_rows.append(\n                row.to_dict()\n            )\n            kept_embeddings.append(\n                current_embedding\n            )\n\n    return pd.DataFrame(\n        kept_rows\n    )\n\n\ndef mmr_select(\n    candidates,\n    top_k=FINAL_TOP_K,\n    lambda_mult=MMR_LAMBDA,\n    max_per_document=MAX_PER_DOCUMENT,\n):\n    if candidates.empty:\n        return candidates.copy()\n\n    candidates = (\n        candidates\n        .sort_values(\n            "effective_relevance",\n            ascending=False,\n        )\n        .reset_index(drop=True)\n    )\n\n    selected_indices = []\n    document_counts = {}\n\n    while (\n        len(selected_indices) < top_k\n        and len(selected_indices)\n        < len(candidates)\n    ):\n        best_index = None\n        best_score = -np.inf\n\n        for idx, row in candidates.iterrows():\n            if idx in selected_indices:\n                continue\n\n            document = row["documento"]\n\n            if (\n                document_counts.get(\n                    document,\n                    0,\n                )\n                >= max_per_document\n            ):\n                continue\n\n            relevance = float(\n                row["effective_relevance"]\n            )\n\n            if not selected_indices:\n                diversity_penalty = 0.0\n            else:\n                current_embedding = row[\n                    "embedding"\n                ]\n\n                diversity_penalty = max(\n                    float(\n                        np.dot(\n                            current_embedding,\n                            candidates.loc[\n                                selected_idx,\n                                "embedding",\n                            ],\n                        )\n                    )\n                    for selected_idx\n                    in selected_indices\n                )\n\n            mmr_score = (\n                lambda_mult\n                * relevance\n                - (\n                    1.0 - lambda_mult\n                )\n                * diversity_penalty\n            )\n\n            if mmr_score > best_score:\n                best_score = mmr_score\n                best_index = idx\n\n        if best_index is None:\n            break\n\n        selected_indices.append(\n            best_index\n        )\n\n        selected_document = candidates.loc[\n            best_index,\n            "documento",\n        ]\n\n        document_counts[\n            selected_document\n        ] = (\n            document_counts.get(\n                selected_document,\n                0,\n            )\n            + 1\n        )\n\n        candidates.loc[\n            best_index,\n            "mmr_score",\n        ] = best_score\n\n    selected = (\n        candidates\n        .loc[selected_indices]\n        .copy()\n        .reset_index(drop=True)\n    )\n\n    selected.insert(\n        0,\n        "rank",\n        np.arange(\n            1,\n            len(selected) + 1,\n        ),\n    )\n\n    return selected\n\n\ndef merge_open_change_continuations(\n    selected,\n    query,\n    document_title,\n):\n    """\n    Si un fragmento seleccionado anuncia que una norma \'queda modificada\n    en los siguientes términos\', concatena el chunk inmediatamente posterior\n    dentro del mismo resultado.\n\n    Así no se desperdicia uno de los 4 puestos con un mero encabezado y,\n    al mismo tiempo, se conserva la continuidad jurídica entre páginas/chunks.\n    """\n    if (\n        selected.empty\n        or not is_change_query(query)\n    ):\n        return selected.copy()\n\n    open_ending_terms = [\n        "queda modificado",\n        "queda modificada",\n        "en los siguientes terminos",\n        "se modifica",\n    ]\n\n    merged = selected.copy()\n\n    for idx, row in merged.iterrows():\n        text_norm = normalize_text(\n            row["texto"]\n        )\n\n        if not any(\n            normalize_text(term)\n            in text_norm\n            for term in open_ending_terms\n        ):\n            continue\n\n        neighbor = get_neighbor_chunk(\n            document_title=document_title,\n            page=int(row["pagina"]),\n            chunk_index=int(\n                row["chunk_index"]\n            ),\n            direction=1,\n        )\n\n        if neighbor is None:\n            continue\n\n        neighbor_text = str(\n            neighbor["text"]\n        ).strip()\n\n        if not neighbor_text:\n            continue\n\n        neighbor_page = int(\n            neighbor["page"]\n        )\n\n        merged.at[\n            idx,\n            "texto",\n        ] = (\n            str(row["texto"]).rstrip()\n            + "\\n\\n"\n            + (\n                f"[Continuación inmediata, "\n                f"página {neighbor_page}]\\n"\n            )\n            + neighbor_text\n        )\n\n        merged.at[\n            idx,\n            "continuation_page",\n        ] = neighbor_page\n\n    return merged\n\n\ndef retrieve_documents(\n    query,\n    n_results=FINAL_TOP_K,\n    min_similarity=MIN_SIMILARITY,\n):\n    """\n    Retrieval híbrido v2.2.\n\n    DOCUMENTO EXPLÍCITO:\n    - consulta filtrada al documento citado;\n    - bonus léxico para preguntas de cambios;\n    - continuidad jurídica: un encabezado de modificación se fusiona\n      con su chunk inmediatamente posterior.\n\n    PREGUNTA GENERAL:\n    - búsqueda semántica;\n    - expansión controlada para impacto residencial;\n    - candidate pool ampliado en esa intención;\n    - bonus léxico temático;\n    - deduplicación + MMR + diversidad documental.\n    """\n    explicit_documents = detect_explicit_documents(\n        query\n    )\n\n    # =====================================================\n    # A. Documento citado explícitamente\n    # =====================================================\n    if explicit_documents:\n        primary_document = explicit_documents[0]\n\n        candidates = retrieve_explicit_document(\n            query=query,\n            document_title=primary_document,\n            n_results=12,\n        )\n\n        if not candidates.empty:\n            candidates = candidates.loc[\n                candidates[\n                    "semantic_similarity"\n                ]\n                >= min_similarity\n            ].copy()\n\n        if candidates.empty:\n            # Fallback general si el filtro documental no recupera\n            # evidencia suficiente.\n            candidates = retrieve_candidates(\n                query\n            )\n\n            if candidates.empty:\n                return candidates\n\n            candidates = candidates.loc[\n                candidates[\n                    "semantic_similarity"\n                ]\n                >= min_similarity\n            ].copy()\n\n            if candidates.empty:\n                return candidates\n\n            selected = mmr_select(\n                remove_near_duplicates(\n                    candidates\n                ),\n                top_k=n_results,\n            )\n\n        else:\n            deduplicated = remove_near_duplicates(\n                candidates\n            )\n\n            selected = mmr_select(\n                deduplicated,\n                top_k=n_results,\n                max_per_document=n_results,\n            )\n\n            selected = (\n                merge_open_change_continuations(\n                    selected=selected,\n                    query=query,\n                    document_title=primary_document,\n                )\n            )\n\n    # =====================================================\n    # B. Pregunta general\n    # =====================================================\n    else:\n        candidates = retrieve_candidates(\n            query\n        )\n\n        if candidates.empty:\n            return candidates\n\n        candidates = candidates.loc[\n            candidates[\n                "semantic_similarity"\n            ]\n            >= min_similarity\n        ].copy()\n\n        if candidates.empty:\n            return candidates\n\n        deduplicated = remove_near_duplicates(\n            candidates\n        )\n\n        selected = mmr_select(\n            deduplicated,\n            top_k=n_results,\n            max_per_document=(\n                MAX_PER_DOCUMENT\n            ),\n        )\n\n    if selected.empty:\n        return selected\n\n    selected = (\n        selected\n        .drop_duplicates(\n            subset=[\n                "documento",\n                "pagina",\n                "chunk_index",\n            ],\n            keep="first",\n        )\n        .reset_index(drop=True)\n        .drop(\n            columns=[\n                "embedding",\n            ],\n            errors="ignore",\n        )\n    )\n\n    selected["rank"] = np.arange(\n        1,\n        len(selected) + 1,\n    )\n\n    columns = [\n        "rank"\n    ] + [\n        column\n        for column in selected.columns\n        if column != "rank"\n    ]\n\n    return selected[columns]\n\ndef format_rag_context(\n    retrieved_df\n):\n    context_parts = []\n    source_map = {}\n\n    for i, row in (\n        retrieved_df.iterrows()\n    ):\n        source_id = f"S{i + 1}"\n\n        source_map[\n            source_id\n        ] = {\n            "documento": row[\n                "documento"\n            ],\n            "pagina": int(\n                row["pagina"]\n            ),\n        }\n\n        similarity = row.get(\n            "semantic_similarity",\n            np.nan,\n        )\n\n        similarity_text = (\n            f"{float(similarity):.3f}"\n            if pd.notna(similarity)\n            else "contexto_vecino"\n        )\n\n        context_parts.append(\n            (\n                f"[{source_id}]\\n"\n                f"FUENTE: {row[\'documento\']}\\n"\n                f"TIPO: {row.get(\'tipo\', \'\')}\\n"\n                f"PÁGINA: {int(row[\'pagina\'])}\\n"\n                f"SIMILITUD: {similarity_text}\\n\\n"\n                f"{row[\'texto\']}"\n            )\n        )\n\n    return (\n        "\\n\\n---\\n\\n".join(\n            context_parts\n        ),\n        source_map,\n    )\n\n\ndef replace_source_ids(\n    answer,\n    source_map,\n):\n    def replacement(match):\n        source_id = match.group(1)\n\n        if source_id not in source_map:\n            return match.group(0)\n\n        source = source_map[\n            source_id\n        ]\n\n        return (\n            f"[{source[\'documento\']}, "\n            f"p. {source[\'pagina\']}]"\n        )\n\n    return re.sub(\n        r"\\[(S\\d+)\\]",\n        replacement,\n        answer,\n    )\n\nrag_prompt = ChatPromptTemplate.from_messages(\n    [\n        (\n            "system",\n            """\nEres un asistente documental especializado en viviendas de uso turístico\n(VUT) en Málaga y Andalucía.\n\nDebes responder ÚNICAMENTE con la información incluida en el CONTEXTO.\n\nPROCEDIMIENTO OBLIGATORIO:\n\nAntes de redactar la respuesta:\n- revisa TODOS los fragmentos recuperados;\n- identifica qué aporta cada fragmento a la pregunta;\n- descarta únicamente los fragmentos que no sean relevantes;\n- no te limites al primer fragmento si otros aportan información\n  complementaria necesaria.\n\nREGLAS:\n\n1. Responde exactamente a la pregunta formulada.\n\n2. Si la pregunta menciona expresamente un documento\n   (por ejemplo, "¿Qué establece la Instrucción 1/2024?"),\n   sintetiza las principales ideas relevantes recuperadas DE ESE\n   DOCUMENTO. No atribuyas al documento principal afirmaciones que\n   procedan de otra fuente.\n\n3. Cuando varios fragmentos del documento solicitado aporten\n   aspectos diferentes y relevantes, intégralos en la respuesta.\n\n4. No añadas información que no aparezca en el contexto.\n\n5. No utilices conocimiento general ni conocimiento previo del modelo.\n\n6. Distingue, cuando sea necesario, entre:\n   - normativa autonómica;\n   - normativa o instrumentos municipales;\n   - planeamiento urbanístico;\n   - informes técnicos o jurídicos.\n\n7. No confundas una norma citada dentro de un documento con\n   el propio contenido o finalidad del documento consultado.\n\n8. Si varias fuentes repiten la misma idea, exprésala una sola vez.\n\n9. Si la evidencia recuperada no permite responder con seguridad,\n   responde exactamente:\n   "La documentación recuperada no aporta evidencia suficiente para\n   responder con precisión a esta pregunta."\n\n10. Cada afirmación importante debe indicar la fuente que la respalda\n    mediante [S1], [S2], [S3] o [S4].\n\n11. Solo puedes utilizar identificadores presentes en el CONTEXTO.\n\n12. No inventes fuentes, artículos, cifras ni consecuencias.\n\n13. Para preguntas amplias, organiza la respuesta en varios puntos\n    breves cuando existan varias ideas diferentes.\n\n14. Evita repetir una conclusión en un párrafo final si ya ha quedado\n    explicada en la respuesta.\n\n15. La respuesta debe ser concisa, pero suficientemente completa\n    para responder a toda la pregunta.\n\n16. La respuesta es informativa y no sustituye asesoramiento jurídico.\n            """,\n        ),\n        (\n            "human",\n            """\nPREGUNTA:\n{question}\n\nCONTEXTO:\n{context}\n\nAnaliza todos los fragmentos relevantes y responde únicamente\na la pregunta formulada.\n            """,\n        ),\n    ]\n)\n\nprint("Prompt v3 preparado")\n\nllm_cache = {}\n\n\ndef get_llm(model_name):\n    if model_name not in llm_cache:\n        llm_cache[\n            model_name\n        ] = ChatOllama(\n            model=model_name,\n            temperature=0,\n        )\n\n    return llm_cache[\n        model_name\n    ]\n\n\ndef ask_rag(\n    question,\n    model_name=FINAL_LLM,\n    n_results=FINAL_TOP_K,\n):\n    retrieved = retrieve_documents(\n        question,\n        n_results=n_results,\n    )\n\n    if retrieved.empty:\n        return {\n            "question": question,\n            "model": model_name,\n            "answer": (\n                "La documentación recuperada no aporta "\n                "evidencia suficiente para responder "\n                "con precisión a esta pregunta."\n            ),\n            "raw_answer": None,\n            "sources": retrieved,\n            "generation_seconds": 0.0,\n            "abstained_before_llm": True,\n        }\n\n    context, source_map = (\n        format_rag_context(\n            retrieved\n        )\n    )\n\n    llm = get_llm(\n        model_name\n    )\n\n    chain = (\n        rag_prompt\n        | llm\n    )\n\n    start = time.perf_counter()\n\n    response = chain.invoke(\n        {\n            "question": question,\n            "context": context,\n        }\n    )\n\n    elapsed = (\n        time.perf_counter()\n        - start\n    )\n\n    raw_answer = str(\n        response.content\n    ).strip()\n\n    final_answer = (\n        replace_source_ids(\n            raw_answer,\n            source_map,\n        )\n    )\n\n    return {\n        "question": question,\n        "model": model_name,\n        "answer": final_answer,\n        "raw_answer": raw_answer,\n        "sources": retrieved,\n        "generation_seconds": elapsed,\n        "abstained_before_llm": False,\n    }\n'

RAG_UTILS_PATH.write_text(
    RAG_UTILS_CODE,
    encoding="utf-8",
)

print("Creado:", RAG_UTILS_PATH)

## 3. Generar `api.py`

`POST /rag` utiliza el `ask_rag()` final y usa 4 fuentes por defecto.

In [ ]:
API_CODE = 'from pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\n\nfrom fastapi import FastAPI, HTTPException\nfrom fastapi.encoders import jsonable_encoder\nfrom pydantic import BaseModel, Field\n\nfrom src.model_utils import predict_listing\nfrom src.rag_utils import ask_rag\n\n\n# ============================================================\n# CONFIGURACIÓN\n# ============================================================\n\nPROJECT_ROOT = Path(__file__).resolve().parent\n\napp = FastAPI(\n    title="VUT Málaga API",\n    description=(\n        "API REST para el motor predictivo XGBoost y el "\n        "consultor documental RAG del TFM."\n    ),\n    version="1.1.0",\n)\n\n\n# ============================================================\n# ESQUEMAS DE ENTRADA\n# ============================================================\n\nclass PredictionRequest(BaseModel):\n    latitude: float = Field(..., ge=-90, le=90)\n    longitude: float = Field(..., ge=-180, le=180)\n\n    room_type: str\n    property_type_group: str\n\n    accommodates: int = Field(..., ge=1, le=16)\n    bedrooms: int = Field(..., ge=0, le=15)\n    bathrooms_num: float = Field(..., ge=0, le=10)\n\n    minimum_nights: int = Field(..., ge=1)\n    maximum_nights: int = Field(..., ge=1)\n\n    instant_bookable: int = Field(..., ge=0, le=1)\n    host_is_superhost: int = Field(..., ge=0, le=1)\n\n    selected_amenities: list[str] = []\n\n\nclass RAGRequest(BaseModel):\n    question: str = Field(..., min_length=3)\n    n_results: int = Field(default=4, ge=1, le=10)\n\n\n# ============================================================\n# UTILIDADES DE SERIALIZACIÓN\n# ============================================================\n\ndef _to_python(value: Any):\n    """Convierte tipos NumPy/Pandas a tipos JSON estándar."""\n\n    if value is None:\n        return None\n\n    if isinstance(value, (np.integer,)):\n        return int(value)\n\n    if isinstance(value, (np.floating,)):\n        if np.isnan(value):\n            return None\n        return float(value)\n\n    if isinstance(value, (np.bool_,)):\n        return bool(value)\n\n    if isinstance(value, pd.Timestamp):\n        return value.isoformat()\n\n    if pd.isna(value):\n        return None\n\n    return value\n\n\ndef _row_to_dict(row: pd.Series) -> dict:\n    return {\n        str(key): _to_python(value)\n        for key, value in row.items()\n    }\n\n\ndef _serialize_shap(explanation) -> dict:\n    values = np.asarray(\n        explanation.values,\n        dtype=float,\n    ).reshape(-1)\n\n    base_value = np.asarray(\n        explanation.base_values,\n        dtype=float,\n    ).reshape(-1)\n\n    feature_names = [\n        str(name)\n        for name in explanation.feature_names\n    ]\n\n    data = np.asarray(\n        explanation.data,\n        dtype=object,\n    ).reshape(-1)\n\n    return {\n        "base_value": float(base_value[0]),\n        "values": [\n            float(value)\n            for value in values\n        ],\n        "feature_names": feature_names,\n        "data": [\n            _to_python(value)\n            for value in data\n        ],\n    }\n\n\n# ============================================================\n# ENDPOINTS\n# ============================================================\n\n@app.get("/health")\ndef health():\n    return {\n        "status": "ok",\n        "service": "VUT Málaga API",\n        "version": "1.1.0",\n    }\n\n\n@app.get("/model-info")\ndef model_info():\n    return {\n        "model": "XGBoost",\n        "target": "price",\n        "objective": "reg:absoluteerror",\n        "validation": "holdout espacial H3 res8 80/20",\n        "metrics": {\n            "r2": 0.529734,\n            "rmse_eur": 57.941030,\n            "mae_eur": 33.202631,\n            "mape_pct": 24.054230,\n        },\n    }\n\n\n@app.post("/predict")\ndef predict(request: PredictionRequest):\n\n    try:\n        result = predict_listing(\n            latitude=request.latitude,\n            longitude=request.longitude,\n            room_type=request.room_type,\n            property_type_group=request.property_type_group,\n            accommodates=request.accommodates,\n            bedrooms=request.bedrooms,\n            bathrooms_num=request.bathrooms_num,\n            minimum_nights=request.minimum_nights,\n            maximum_nights=request.maximum_nights,\n            instant_bookable=request.instant_bookable,\n            host_is_superhost=request.host_is_superhost,\n            selected_amenities=request.selected_amenities,\n        )\n\n        feature_row = result["features"].iloc[0]\n\n        response = {\n            "price": float(result["price"]),\n            "features": _row_to_dict(feature_row),\n            "shap": _serialize_shap(\n                result["shap_explanation"]\n            ),\n        }\n\n        return jsonable_encoder(response)\n\n    except Exception as exc:\n        raise HTTPException(\n            status_code=500,\n            detail=f"Error en la predicción: {exc}",\n        ) from exc\n\n\n@app.post("/rag")\ndef rag(request: RAGRequest):\n\n    try:\n        result = ask_rag(\n            request.question,\n            n_results=request.n_results,\n        )\n\n        sources = result["sources"]\n\n        if sources is None:\n            source_records = []\n\n        elif isinstance(sources, pd.DataFrame):\n            source_records = [\n                {\n                    str(key): _to_python(value)\n                    for key, value in row.items()\n                }\n                for row in sources.to_dict(\n                    orient="records"\n                )\n            ]\n\n        else:\n            source_records = sources\n\n        response = {\n            "question": request.question,\n            "answer": result["answer"],\n            "sources": source_records,\n        }\n\n        return jsonable_encoder(response)\n\n    except Exception as exc:\n        raise HTTPException(\n            status_code=500,\n            detail=f"Error en el sistema RAG: {exc}",\n        ) from exc\n'

API_PATH.write_text(
    API_CODE,
    encoding="utf-8",
)

print("Creado:", API_PATH)

## 4. Generar `src/api_client.py`



In [ ]:
API_CLIENT_CODE = 'import os\n\nimport requests\n\n\nAPI_BASE_URL = os.getenv(\n    "VUT_API_URL",\n    "http://127.0.0.1:8000",\n)\n\n\ndef health_api(timeout=10):\n    response = requests.get(\n        f"{API_BASE_URL}/health",\n        timeout=timeout,\n    )\n    response.raise_for_status()\n    return response.json()\n\n\ndef model_info_api(timeout=10):\n    response = requests.get(\n        f"{API_BASE_URL}/model-info",\n        timeout=timeout,\n    )\n    response.raise_for_status()\n    return response.json()\n\n\ndef predict_api(payload, timeout=120):\n    response = requests.post(\n        f"{API_BASE_URL}/predict",\n        json=payload,\n        timeout=timeout,\n    )\n    response.raise_for_status()\n    return response.json()\n\n\ndef rag_api(\n    question,\n    n_results=4,\n    timeout=900,\n):\n    response = requests.post(\n        f"{API_BASE_URL}/rag",\n        json={\n            "question": question,\n            "n_results": n_results,\n        },\n        timeout=timeout,\n    )\n    response.raise_for_status()\n    return response.json()\n'

API_CLIENT_PATH.write_text(
    API_CLIENT_CODE,
    encoding="utf-8",
)

print("Creado:", API_CLIENT_PATH)

## 5. Validación estática

In [ ]:
import ast

for path in [
    RAG_UTILS_PATH,
    API_PATH,
    API_CLIENT_PATH,
]:
    source = path.read_text(
        encoding="utf-8"
    )

    ast.parse(source)

    print(
        f"OK sintaxis: {path.name}"
    )

## 6. Dependencias

Las dependencias necesarias para ejecutar la API y el resto de componentes
del proyecto se recogen en el archivo `requirements.txt` situado en la raíz.

## 7. Arranque de FastAPI

Con los archivos de producción ya disponibles en el proyecto, la API puede
arrancarse desde un CMD abierto en la raíz mediante:

```bat
"....\anaconda3\python.exe" -m uvicorn api:app --host 127.0.0.1 --port 8000
```

## 8. Prueba `/health`

Ejecutar solo cuando Uvicorn esté arrancado.

In [ ]:
from src.api_client import health_api

health_result = health_api()

health_result

## 9. Prueba `/model-info`

In [ ]:
from src.api_client import model_info_api

model_info_result = model_info_api()

model_info_result

## 10. Prueba `/predict`

In [ ]:
from src.api_client import predict_api

prediction_payload = {
    "latitude": 36.7200,
    "longitude": -4.4200,
    "room_type": "Entire home/apt",
    "property_type_group": "Apartamento",
    "accommodates": 4,
    "bedrooms": 2,
    "bathrooms_num": 1.0,
    "minimum_nights": 2,
    "maximum_nights": 365,
    "instant_bookable": 1,
    "host_is_superhost": 0,
    "selected_amenities": [
        "has_wifi",
        "has_kitchen",
        "has_air_conditioning",
    ],
}

prediction_api_result = predict_api(
    prediction_payload
)

print(
    "Tarifa API:",
    round(
        prediction_api_result["price"],
        2,
    ),
    "€ / noche",
)

print(
    "Distrito:",
    prediction_api_result[
        "features"
    ].get("district"),
)

print(
    "Nº contribuciones SHAP:",
    len(
        prediction_api_result[
            "shap"
        ]["values"]
    ),
)


## 11. Prueba `/rag` final



In [ ]:
from src.api_client import rag_api

rag_api_result = rag_api(
    (
        "¿Qué ocurre cuando un barrio supera "
        "el 8% de viviendas de uso turístico?"
    ),
    n_results=4,
)

print(
    rag_api_result["answer"]
)

print("\nFuentes:")

for source in rag_api_result["sources"]:
    print(
        "-",
        source.get("documento"),
        "· página",
        source.get("pagina"),
    )

# Resultado


```text
Streamlit
   │
   ▼
FastAPI
├── /predict ──► XGBoost + SHAP
└── /rag ─────► RAG v2.2 + prompt v3 + Mistral 7B
```